# Build Porseman training dataset

Create one positive answer and seven LLM-reviewed hard negatives for each training question. This notebook reads only the training split; test and validation data remain untouched.

## Setup

Locate the project root so the notebook works when opened from either the repository root or the `notebooks` directory.

In [ ]:
import csv
from pathlib import Path

def find_project_root(start_path):
    for candidate in (start_path, *start_path.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root. Run this notebook from inside the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAIN_INPUT_PATH = PROJECT_ROOT / "data/processed/porseman_train.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_train_with_hard_negatives.jsonl"

RETRIEVAL_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RETRIEVAL_CANDIDATE_COUNT = 100
RERANKER_CANDIDATE_COUNT = 20
NEGATIVES_PER_QUERY = 7

print(f"Training input: {TRAIN_INPUT_PATH}")
print(f"Output dataset: {OUTPUT_PATH}")

Training input: C:\Users\1\Documents\finetune-embedding-models\data\processed\porseman_train.csv
Output dataset: C:\Users\1\Documents\finetune-embedding-models\data\processed\porseman_train_with_hard_negatives.jsonl


## 1. Read the training split

Load the train split and verify the fields required for hard-negative mining. Every row retains its stable ID, question, and positive answer.

In [ ]:
REQUIRED_FIELDS = {"id", "question", "content_text"}

if not TRAIN_INPUT_PATH.is_file():
    raise FileNotFoundError(f"Training CSV was not found: {TRAIN_INPUT_PATH}")

with TRAIN_INPUT_PATH.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    missing_fields = REQUIRED_FIELDS - set(reader.fieldnames or [])
    if missing_fields:
        raise ValueError(f"Training CSV is missing fields: {sorted(missing_fields)}")
    train_rows = [
        {field: (row[field] or "").strip() for field in REQUIRED_FIELDS}
        for row in reader
    ]

invalid_rows = [row for row in train_rows if not all(row.values())]
if invalid_rows:
    raise ValueError(f"Training CSV has {len(invalid_rows):,} row(s) with an empty required field.")

train_ids = [row["id"] for row in train_rows]
train_queries = [row["question"] for row in train_rows]
train_positives = [row["content_text"] for row in train_rows]

if len(train_ids) != len(set(train_ids)):
    raise ValueError("Training CSV has duplicate IDs.")

print(f"Training rows: {len(train_rows):,}")
print(f"Unique positive answers: {len(set(train_positives)):,}")

Training rows: 13,880
Unique positive answers: 13,615
